# Benchmark Contamination

If a benchmark's questions were in the training data, the score measures memorisation
rather than capability. Given that pre-training corpora are scraped from the same web
that hosts every public benchmark, the default assumption for any widely-used benchmark
should be that **some contamination is present** — the open question is how much, and
whether it changed the conclusion.

This is not a hypothetical failure mode. It is the most likely explanation for a
surprisingly high score, and it is the reason a model can top a leaderboard and then
disappoint in use.

Completes the evaluation track: [Eval Harness Design](eval-harness-design.ipynb),
[LLM-as-a-Judge](llm-as-judge.ipynb), [The Statistics of Evals](eval-statistics.ipynb).
Detection here shares its machinery with
[Data Deduplication](../14-pretraining/data-deduplication.ipynb).

## 1. What & Why

Contamination comes in degrees, and they are not equally damaging:

| Kind | What leaked | Effect |
|---|---|---|
| **Verbatim input+label** | The question *and* its answer | Severe. The model can recall the answer |
| **Input only** | The question, no answer | Mild but real — familiarity with phrasing helps |
| **Derived / discussion** | A blog post analysing the benchmark's questions | Moderate, and very hard to detect |
| **Distributional** | Same *style* of question from the same source | Not contamination; this is legitimate generalisation |
| **Indirect** | Another model's outputs on the benchmark, in your training data | Increasingly common, and largely invisible |

**Why it is hard to escape:** benchmarks are published on GitHub, mirrored on Hugging
Face, copied into blog posts, and discussed on Stack Exchange. A crawl of the web in 2024
contains most benchmarks published before 2024, several times over.

**Why it matters beyond leaderboards:** contamination inflates the specific numbers you
use to make decisions. If you pick training recipe A over B because A scored higher on a
contaminated benchmark, you may have selected for memorisation of that benchmark.

## 2. Mental Model

**An exam where some questions leaked, and you do not know which.**

That framing gets three things right that "the model cheated" does not:

- **It is a spectrum, not a binary.** A student who saw three of fifty questions is not
  the same as one who saw all fifty. The useful quantity is *how much of the score* is
  attributable to leakage, not whether any leaked.
- **The student may not know either.** A model has no notion of which items it saw in
  training. There is no intent, and asking the model is not a reliable detector.
- **The fix is a fresh exam, not a better invigilator.** Detection is a stopgap.
  The only robust answer is items the model cannot have seen — a **held-out or
  post-cutoff** benchmark.

The diagnostic that follows from the analogy: a student who genuinely knows the material
performs similarly on old and new questions of the same difficulty. **A gap between
performance on pre-cutoff and post-cutoff items, holding difficulty fixed, is the
signature of contamination** — and it is measurable without access to the training data.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **n-gram overlap** | The standard detector: flag a test item if a long enough n-gram (typically 8–13 tokens) appears in the training corpus. |
| **Decontamination** | Removing flagged items from *training* (better) or from the *eval* (weaker, changes the benchmark). |
| **Canary string** | A unique GUID published inside a benchmark file so anyone can grep a corpus — or ask a model to reproduce it — to prove exposure. |
| **Post-cutoff evaluation** | Items created after the model's training cutoff. The strongest available guarantee. |
| **Temporal split** | Comparing accuracy on items before versus after the cutoff, holding difficulty fixed. |
| **Membership inference** | Inferring whether an example was trained on, e.g. via unusually low perplexity relative to comparable text. |
| **Perturbation / paraphrase test** | Rewrite items while preserving the answer. A memorising model degrades sharply; a capable one does not. |
| **Option-order sensitivity** | A model that memorised "the answer is C" fails when options are shuffled. |
| **Indirect contamination** | Leakage via another model's outputs, or via discussion rather than the data itself. |
| **Distributional similarity** | Legitimately learning the task's *style*. Not contamination, and the distinction is a judgement call. |

## 4. Setup

NumPy and the standard library. As throughout this track, the contamination is simulated
so its true extent is known and each detector can be scored against ground truth — you
can see which ones work and which quietly do not.

In [1]:
# %pip install numpy

import hashlib
import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — n-gram overlap detection, and picking `n`

The workhorse detector. The parameter that decides whether it works is the n-gram length:
too short and every item collides on common phrasing, too long and paraphrase evades it.

In [2]:
COMMON = ("the model was trained on a large corpus of text and then evaluated "
          "carefully on a standard benchmark to measure its performance").split()

def make_item(seed):
    r = np.random.default_rng(seed)
    filler = [f"w{r.integers(0, 400)}" for _ in range(14)]
    # every item shares some boilerplate phrasing -- this is what causes false positives
    return " ".join(COMMON[:6] + filler)

test_items = [make_item(i) for i in range(300)]
CONTAMINATED = set(range(0, 300, 6))          # every 6th item leaked verbatim

# The "training corpus": the leaked items, plus unrelated text.
corpus_docs = [test_items[i] for i in sorted(CONTAMINATED)]
corpus_docs += [make_item(10_000 + i) for i in range(3000)]

def ngrams(text, n):
    toks = text.split()
    return {" ".join(toks[i:i + n]) for i in range(len(toks) - n + 1)}

def build_index(docs, n):
    idx = set()
    for d in docs:
        idx |= ngrams(d, n)
    return idx

print(f"{'n':>4} {'flagged':>8} {'true pos':>9} {'false pos':>10} {'missed':>7} "
      f"{'precision':>10} {'recall':>8}")
for n in (3, 5, 8, 13, 20):
    idx = build_index(corpus_docs, n)
    flagged = {i for i, t in enumerate(test_items) if ngrams(t, n) & idx}
    tp = len(flagged & CONTAMINATED)
    fp = len(flagged - CONTAMINATED)
    fn = len(CONTAMINATED - flagged)
    prec = tp / len(flagged) if flagged else float("nan")
    print(f"{n:>4} {len(flagged):8d} {tp:9d} {fp:10d} {fn:7d} {prec:10.2f} "
          f"{tp/len(CONTAMINATED):8.2f}")

print("\nShort n-grams flag everything -- shared boilerplate is enough to collide.")
print("Long n-grams are precise here, which is why 8-13 is the usual choice.")
print("\nBut note what this detector fundamentally cannot see: it needs access to the")
print("training corpus. For any model you did not train, it is unavailable, which is")
print("why the behavioural tests below matter.")

   n  flagged  true pos  false pos  missed  precision   recall
   3      300        50        250       0       0.17     1.00
   5      300        50        250       0       0.17     1.00
   8       53        50          3       0       0.94     1.00
  13       50        50          0       0       1.00     1.00
  20       50        50          0       0       1.00     1.00

Short n-grams flag everything -- shared boilerplate is enough to collide.
Long n-grams are precise here, which is why 8-13 is the usual choice.

But note what this detector fundamentally cannot see: it needs access to the
training corpus. For any model you did not train, it is unavailable, which is
why the behavioural tests below matter.


### Example 2 — paraphrase evades n-gram detection entirely

The detector's blind spot, demonstrated: an item that leaked in *reworded* form still
helps the model, and no exact n-gram matcher will find it.

In [3]:
original = "what is the capital city of the country of france in western europe"
variants = {
    "verbatim":        original,
    "word swapped":    "what is the capital town of the country of france in western europe",
    "reordered":       "in western europe what is the capital city of the country of france",
    "paraphrased":     "name the city that serves as the seat of government for france",
}

idx13 = ngrams(original, 13)
idx8 = ngrams(original, 8)
print(f"{'variant':16} {'13-gram hit':>13} {'8-gram hit':>12}")
for name, text in variants.items():
    print(f"{name:16} {str(bool(ngrams(text, 13) & idx13)):>13} "
          f"{str(bool(ngrams(text, 8) & idx8)):>12}")

print("\nA single word substitution defeats 13-gram matching. Reordering defeats both.")
print("A genuine paraphrase is invisible to any exact-match detector.")
print("\nThis is not a flaw to be tuned away: exact matching cannot detect semantic")
print("leakage, and semantic leakage still teaches the model the answer. Treat an")
print("n-gram report as a LOWER BOUND on contamination, never as a clearance.")

variant            13-gram hit   8-gram hit
verbatim                  True         True
word swapped             False         True
reordered                False         True
paraphrased              False        False

A single word substitution defeats 13-gram matching. Reordering defeats both.
A genuine paraphrase is invisible to any exact-match detector.

This is not a flaw to be tuned away: exact matching cannot detect semantic
leakage, and semantic leakage still teaches the model the answer. Treat an
n-gram report as a LOWER BOUND on contamination, never as a clearance.


### Example 3 — the temporal split: detection without the training data

The test that works on a model you did not train. Split the benchmark by item creation
date around the model's cutoff. A model that *understands* performs the same on both
sides; a model that *memorised* does not.

The whole validity of this test rests on the two halves being equally difficult, so the
example checks that assumption rather than assuming it.

In [4]:
n_pre, n_post = 600, 600
# Same difficulty distribution on both sides -- the assumption the test depends on.
diff_pre = rng.normal(0, 1, n_pre)
diff_post = rng.normal(0, 1, n_post)

def accuracy(difficulty, ability, memorised_frac=0.0, seed=0):
    r = np.random.default_rng(seed)
    p = 1 / (1 + np.exp(difficulty - ability))
    memorised = r.random(len(difficulty)) < memorised_frac
    p = np.where(memorised, 0.98, p)            # a recalled item is nearly always right
    return float(np.mean(r.random(len(difficulty)) < p))

print(f"{'model':34} {'pre-cutoff':>11} {'post-cutoff':>12} {'gap':>7}")
for name, mem in [("clean (no contamination)", 0.00),
                  ("lightly contaminated", 0.15),
                  ("heavily contaminated", 0.45)]:
    a_pre = accuracy(diff_pre, 0.3, mem, seed=1)
    a_post = accuracy(diff_post, 0.3, 0.0, seed=2)     # post-cutoff cannot be memorised
    print(f"{name:34} {a_pre:11.1%} {a_post:12.1%} {a_pre - a_post:+7.1%}")

print("\nA large positive gap is the signature. The clean model shows ~0.")

print("\nThe assumption this test rests on, checked:")
print(f"  mean difficulty pre  = {diff_pre.mean():+.3f}")
print(f"  mean difficulty post = {diff_post.mean():+.3f}")
print("\nIf post-cutoff items are HARDER (very common -- recent benchmarks are built to")
print("be hard, precisely because models solved the old ones), the gap appears without")
print("any contamination at all. Matching difficulty across the split is the entire")
print("methodological difficulty of this test, and it is why it is best applied to")
print("naturally-dated item streams -- competition problems, exam papers, news -- rather")
print("than to two benchmarks written years apart.")

model                               pre-cutoff  post-cutoff     gap
clean (no contamination)                 55.0%        55.5%   -0.5%
lightly contaminated                     61.0%        55.5%   +5.5%
heavily contaminated                     75.7%        55.5%  +20.2%

A large positive gap is the signature. The clean model shows ~0.

The assumption this test rests on, checked:
  mean difficulty pre  = -0.023
  mean difficulty post = -0.041

If post-cutoff items are HARDER (very common -- recent benchmarks are built to
be hard, precisely because models solved the old ones), the gap appears without
any contamination at all. Matching difficulty across the split is the entire
methodological difficulty of this test, and it is why it is best applied to
naturally-dated item streams -- competition problems, exam papers, news -- rather
than to two benchmarks written years apart.


### Example 4 — behavioural probes: perturbation and option-order

Two cheap tests that need no training-data access and no dated items. Both work by
changing something that should not matter and checking whether the score moves.

In [5]:
def simulate_mcq(n, ability, memorised_frac, shuffle_options, seed=0):
    '''Memorised items are recalled as a LETTER, so shuffling the options breaks them
    while genuine understanding is unaffected.'''
    r = np.random.default_rng(seed)
    p_understood = 1 / (1 + np.exp(r.normal(0, 1, n) - ability))
    memorised = r.random(n) < memorised_frac
    p = np.where(memorised, 0.25 if shuffle_options else 0.98, p_understood)
    return float(np.mean(r.random(n) < p))

print("TEST 1 -- shuffle the answer options (answer content unchanged):\n")
print(f"{'model':28} {'original':>10} {'shuffled':>10} {'drop':>8}")
for name, mem in [("clean", 0.0), ("30% memorised", 0.30), ("60% memorised", 0.60)]:
    a = simulate_mcq(2000, 0.4, mem, shuffle_options=False, seed=7)
    b = simulate_mcq(2000, 0.4, mem, shuffle_options=True, seed=7)
    print(f"{name:28} {a:10.1%} {b:10.1%} {a - b:+8.1%}")

print("\nA model that understands is indifferent to option order. A model that recalled")
print("'the answer is C' collapses toward chance.\n")

print("TEST 2 -- paraphrase the questions (meaning and answer preserved):\n")
def simulate_paraphrase(n, ability, memorised_frac, paraphrased, seed=0):
    r = np.random.default_rng(seed)
    p_understood = 1 / (1 + np.exp(r.normal(0, 1, n) - ability))
    memorised = r.random(n) < memorised_frac
    recall_p = 0.35 if paraphrased else 0.98      # recall is surface-form dependent
    p = np.where(memorised, recall_p, p_understood * (0.97 if paraphrased else 1.0))
    return float(np.mean(r.random(n) < p))

print(f"{'model':28} {'original':>10} {'paraphrased':>12} {'drop':>8}")
for name, mem in [("clean", 0.0), ("30% memorised", 0.30), ("60% memorised", 0.60)]:
    a = simulate_paraphrase(2000, 0.4, mem, False, seed=11)
    b = simulate_paraphrase(2000, 0.4, mem, True, seed=11)
    print(f"{name:28} {a:10.1%} {b:12.1%} {a - b:+8.1%}")

print("\nThe clean model loses a couple of points to paraphrasing (rewrites are never")
print("perfectly meaning-preserving); the contaminated ones lose far more.")
print("\nThat small clean-model drop is why these are COMPARATIVE tests. The absolute")
print("drop means little; the drop relative to a model you believe is clean means a lot.")

TEST 1 -- shuffle the answer options (answer content unchanged):

model                          original   shuffled     drop
clean                             61.6%      61.6%    +0.0%
30% memorised                     71.5%      50.7%   +20.8%
60% memorised                     82.7%      40.9%   +41.8%

A model that understands is indifferent to option order. A model that recalled
'the answer is C' collapses toward chance.

TEST 2 -- paraphrase the questions (meaning and answer preserved):

model                          original  paraphrased     drop
clean                             57.1%        55.8%    +1.4%
30% memorised                     69.7%        49.6%   +20.1%
60% memorised                     81.6%        43.1%   +38.4%

The clean model loses a couple of points to paraphrasing (rewrites are never
perfectly meaning-preserving); the contaminated ones lose far more.

That small clean-model drop is why these are COMPARATIVE tests. The absolute
drop means little; the drop re

### Example 5 — canary strings

A benchmark can make its own exposure detectable by embedding a unique, otherwise
meaningless string in its distribution files. Anyone can then grep a corpus for it, or
ask a model to complete it.

In [6]:
BENCHMARK_ID = "praxis-example-benchmark-v1"
canary = hashlib.sha256(BENCHMARK_ID.encode()).hexdigest()
print("canary GUID published inside the benchmark files:")
print(" ", canary)

print("\nA corpus grep gives a definitive answer for data you control:")
for name, corpus in [("clean crawl", ["ordinary web text", "more web text"]),
                     ("crawl including the benchmark repo",
                      ["ordinary web text", f"BENCHMARK CANARY {canary} test.jsonl"])]:
    hit = any(canary in doc for doc in corpus)
    print(f"  {name:38} canary present: {hit}")

print("\nStrengths and limits, both worth being precise about:")
print("  + A canary hit is PROOF of exposure -- the string has no other reason to exist.")
print("  + It works even when the benchmark text itself was paraphrased in transit.")
print("  - A MISS proves nothing: the benchmark may have been copied without the canary,")
print("    reformatted, or discussed rather than reproduced.")
print("  - It only helps benchmarks that adopted a canary before publication.")
print("\nSo canaries are a one-sided test: useful for establishing contamination,")
print("useless for establishing cleanliness. That asymmetry runs through every")
print("detector in this notebook.")

canary GUID published inside the benchmark files:
  09b2a9addd0b62459edd3467183d666d0645f87c0222aaa8294943d7a0af916b

A corpus grep gives a definitive answer for data you control:
  clean crawl                            canary present: False
  crawl including the benchmark repo     canary present: True

Strengths and limits, both worth being precise about:
  + A canary hit is PROOF of exposure -- the string has no other reason to exist.
  + It works even when the benchmark text itself was paraphrased in transit.
  - A MISS proves nothing: the benchmark may have been copied without the canary,
    reformatted, or discussed rather than reproduced.
  - It only helps benchmarks that adopted a canary before publication.

So canaries are a one-sided test: useful for establishing contamination,
useless for establishing cleanliness. That asymmetry runs through every
detector in this notebook.


## 6. Gotchas & Pitfalls

- **Treating a negative detection result as a clean bill of health.** Every detector here
  is one-sided: a hit proves contamination, a miss proves nothing. Examples 2 and 5.
- **Removing flagged items from the *eval*.** This changes what the benchmark measures —
  and it preferentially removes the items the model has seen, which may be the easy ones.
  Decontaminating the *training set* is the sound direction.
- **Comparing pre/post-cutoff sets of different difficulty.** Example 3. Recent
  benchmarks are usually built to be harder, which manufactures a contamination signal
  from nothing.
- **Asking the model whether it has seen the item.** Models confabulate both ways. This is
  not a detector.
- **Ignoring indirect contamination.** Training on another model's outputs can transfer
  benchmark knowledge with no textual overlap at all.
- **Assuming a private benchmark stays private.** Sending items through a third-party API
  may put them in someone's training data. If it must stay clean, run it locally.
- **Using the same held-out set indefinitely.** Its value decays with every use, whether
  or not it leaks — you optimise against it. Rotate.
- **Confusing distributional similarity with contamination.** A model trained on lots of
  maths will do well on maths benchmarks. That is the point, not cheating. The line is
  whether the *specific items* were seen.
- **Reporting contaminated numbers without saying so.** If you know a benchmark is
  compromised for your model, the number needs an asterisk, not a footnote.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| You control the training data | **n-gram decontamination** at 8–13 tokens, before training |
| You did not train the model | **Temporal split** (Example 3) and **behavioural probes** (Example 4) |
| You are publishing a benchmark | **Canary strings**, plus a held-out private split |
| You need a number you can trust now | **Post-cutoff items** — freshly written, or from a dated stream |
| Detecting near-duplicates at scale | MinHash/LSH — see [Data Deduplication](../14-pretraining/data-deduplication.ipynb) |
| Deciding whether a model is genuinely better | Run it on something with no public existence, and see |

**The honest position.** Detection is damage control. Every method here has a blind spot,
and the blind spots overlap in exactly the case you most care about — semantic leakage of
a widely-discussed benchmark. The methods are still worth running, because a positive
result is decisive and cheap to obtain.

But the only robust answer is **evaluation on data the model cannot have seen**: items
you wrote, items from after the cutoff, or a private set that has never touched a network.
That is expensive and it decays with use, which is why the practical discipline is to keep
a small, fresh, private set for the decisions that matter, and treat public leaderboard
numbers as a weak signal about the models you did not train.

## 8. Resources

- [Documenting Large Webtext Corpora: A Case Study on the Colossal Clean Crawled Corpus](https://arxiv.org/abs/2104.08758) — measures benchmark contamination in C4 directly.
- [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) — Section 4 is the original large-scale n-gram contamination analysis, including what the authors could not clean.
- [Rethinking Benchmark and Contamination for Language Models with Rephrased Samples](https://arxiv.org/abs/2311.04850) — the Example 2 result at scale: rephrased contamination evades n-gram and embedding detectors alike.
- [Investigating Data Contamination in Modern Benchmarks for Large Language Models](https://arxiv.org/abs/2311.09783) — practical probes usable without training-data access.
- [GSM1k: A Careful Examination of Large Language Model Performance on Grade School Arithmetic](https://arxiv.org/abs/2405.00332) — a fresh benchmark built to mirror GSM8K; the accuracy drops are the temporal-split argument in practice.
- [LiveBench](https://livebench.ai/) — a contamination-limited benchmark using continuously refreshed, post-cutoff questions.
- [The BIG-bench canary](https://github.com/google/BIG-bench#kbd-warning-kbd-do-not-include-this-canary-guid-in-your-training-data) — the canary-string convention of Example 5, as actually deployed.